# Agent evaluation — avaliação manual

Execute os dez casos e confira a trajetória no `call_log`.
As definições do agente estão mantidas abaixo para este notebook funcionar sozinho.
A parte nova começa em **Avaliação: dez casos**.


## **Imports**

In [3]:
from copy import deepcopy
from typing import Annotated
from unittest.mock import patch
import json

import httpx
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from bibliometric_rag_assistant.ingestion.crossref import (
    fetch_crossref_work,
    parse_crossref_work,
)
from bibliometric_rag_assistant.ingestion.pubmed import (
    fetch_pubmed_citations,
    parse_pubmed_xml,
    search_pubmed,
)
from bibliometric_rag_assistant.search_strategy import PUBMED_QUERY, PUBMED_SAMPLE_SIZE

load_dotenv()

MODEL = "gpt-5.4-mini"

openai_client = OpenAI()

## Agente avaliado

Estas são as definições do agente já estudadas em `01_typed_tools.ipynb`.
Execute-as antes das avaliações.


Contratos das tools:

In [4]:
Pmid = Annotated[str, Field(pattern=r"^[0-9]+$")]

class PubMedSearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    query: str = Field(
        min_length=1,
        description=(
            "PubMed query using Boolean operators and field tags, "
            "such as [Title/Abstract] and [MeSH Terms]."
        ),
    )

    limit: int = Field(
        ge=1,
        le=10_000,
        description="Número máximo de PMIDs requisitados para esta busca. ",
    )


class PubMedCitationsArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    pmids: list[Pmid] = Field(
        min_length=1,
        description="PMIDs fornecidos pelo usuário ou retornados pela busca.",
    )


class CrossrefWorkArgs(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        strict=True,
        str_strip_whitespace=True,
    )

    doi: str = Field(
        min_length=1,
        description=(
            "DOI fornecido pelo usuário ou obtido nos metadados. "
            "Informe somente o DOI, sem o prefixo https://doi.org/."
        ),
    )

Funções que produzem as observações entregues ao agente:

In [5]:
def run_pubmed_search(query: str, limit: int) -> dict:
    raw_results = search_pubmed(query=query, limit=limit)
    esearch = raw_results["esearchresult"]

    return {
        "status": "success",
        "source": "pubmed",
        "query": query,
        "requested_count": limit,
        "total_results": int(esearch["count"]),
        "returned_count": len(esearch["idlist"]),
        "pmids": esearch["idlist"],
    }


def get_pubmed_citations(pmids: list[str]) -> dict:
    xml = fetch_pubmed_citations(pmids)
    citations = parse_pubmed_xml(xml)

    returned_pmids = {record["pmid"] for record in citations}

    return {
        "status": "success",
        "source": "pubmed",
        "requested_pmids": pmids,
        "returned_count": len(citations),
        "missing_pmids": [
            pmid for pmid in pmids
            if pmid not in returned_pmids
        ],
        "citations": citations,
    }


def get_crossref_work(doi: str) -> dict:
    try:
        payload = fetch_crossref_work(doi)

    except httpx.HTTPStatusError as error:
        if error.response.status_code != 404:
            raise

        return {
            "status": "not_found",
            "source": "crossref",
            "doi": doi,
        }

    return {
        "status": "success",
        "source": "crossref",
        "work": parse_crossref_work(payload["message"]),
    }

Registro das tools e descrição dos contratos:

In [6]:
TOOL_REGISTRY = {
    "search_pubmed": {
        "args_model": PubMedSearchArgs,
        "handler": run_pubmed_search,
        "description": (
            "Search PubMed and return the total number of matching records "
            "and a limited list of PMIDs. "
            "Use this to discover records; obtain publication details "
            "with a separate metadata tool."
        ),
    },
    "get_pubmed_citations": {
        "args_model": PubMedCitationsArgs,
        "handler": get_pubmed_citations,
        "description": (
            "Consulte metadados de PMIDs conhecidos no PubMed. "
            "Retorna título, DOI, periódico e ano de publicação. "
            "Use para obter detalhes dos registros encontrados na busca."
        ),
    },
    "get_crossref_work": {
        "args_model": CrossrefWorkArgs,
        "handler": get_crossref_work,
        "description": (
            "Consulte um DOI no Crossref. Retorna metadados e as contagens "
            "de referências e citações disponíveis nessa fonte. "
            "Requer um DOI fornecido pelo usuário ou obtido de outra ferramenta."
        ),
    },
}

TOOLS = [
    {
        "type": "function",
        "name": name,
        "description": spec["description"],
        "parameters": spec["args_model"].model_json_schema(),
        "strict": True,
    }
    for name, spec in TOOL_REGISTRY.items()
]

Executor:

In [7]:
def make_call(call):
    if call.name not in TOOL_REGISTRY:
        raise ValueError(f"Ferramenta desconhecida: {call.name}")

    spec = TOOL_REGISTRY[call.name]

    try:
        args = spec["args_model"].model_validate_json(call.arguments)

    except ValidationError as error:
        result = {
            "status": "invalid_arguments",
            "errors": [
                {
                    "field": list(item["loc"]),
                    "message": item["msg"],
                }
                for item in error.errors()
            ],
        }

    else:
        try:
            result = spec["handler"](**args.model_dump())

        except httpx.HTTPStatusError as error:
            result = {
                "status": "external_error",
                "tool": call.name,
                "http_status": error.response.status_code,
                "message": "O serviço retornou um erro HTTP.",
            }

        except httpx.RequestError:
            result = {
                "status": "external_error",
                "tool": call.name,
                "message": "Não foi possível concluir a conexão com o serviço.",
            }

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, ensure_ascii=False),
    }

Instruções e agentic loop:

In [8]:
instructions = f"""
Você auxilia consultas bibliográficas no PubMed.

A query atual do projeto é:
{PUBMED_QUERY}

Orientações:
- Use como limit a quantidade explicitamente solicitada pelo usuário.
- Quando nenhuma quantidade for indicada, use {PUBMED_SAMPLE_SIZE}.
- Respeite os limites técnicos descritos no schema da ferramenta.
- Diferencie quantidade solicitada, total encontrado e quantidade retornada.
- Se a quantidade solicitada superar o total encontrado, explique que
  a consulta encontrou menos registros do que o solicitado.
- Baseie os números e PMIDs exclusivamente no resultado da ferramenta.
- Ao receber status invalid_arguments, examine os erros e corrija os
  argumentos antes de tentar novamente, respeitando o orçamento de chamadas.
- Para obter detalhes de um PMID, use get_pubmed_citations.
- Para consultar o Crossref, utilize um DOI fornecido pelo usuário
  ou retornado por uma ferramenta. Nunca invente um DOI.
- Quando o DOI estiver ausente ou o Crossref retornar not_found,
  explique essa ausência.
- Diferencie contagem de referências de contagem de citações e
  identifique o Crossref como fonte dessas contagens.
- Se houver external_error, explique que a consulta não foi concluída.
"""

In [9]:
MAX_TOOL_CALLS = 4


def run_agent(question: str) -> dict:
    history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    call_log = []
    stop_reason = None
    tool_call_count = 0
    final_answer = None
    it = 0

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=MODEL,
            input=history,
            tools=TOOLS,
            parallel_tool_calls=False,
        )
        history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                if tool_call_count >= MAX_TOOL_CALLS:
                    stop_reason = 'max_tool_calls'
                    break

                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                tool_call_count += 1

                call_log.append({
                    'tool': item.name,
                    'arguments': json.loads(item.arguments),
                    'result': json.loads(call_output["output"]),
                })

                history.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print('ASSISTANT:')
                print(final_answer)

        if stop_reason == 'max_tool_calls':
            print(f'Parando: limite de {MAX_TOOL_CALLS} chamadas de tool atingido.')
            break

        it += 1
        if not has_function_calls:
            stop_reason = 'end_turn'
            break

    return {
        'answer': final_answer,
        'call_log': call_log,
        'stop_reason': stop_reason,
    }

## Avaliação: dez casos

Execute um caso por vez. Para cada um, confira:

1. O agente escolheu ferramentas e argumentos adequados?
2. A resposta corresponde aos dados das observações, incluindo ausências e erros?
3. O agente encerrou respeitando o orçamento?

Os casos 01–05 usam as ferramentas reais. Nos casos 06–08 e 10, uma função curta simula o resultado ou a falha de uma ferramenta. O `patch.dict` restaura o handler original ao terminar o bloco `with`.

O caso 09 avalia um **pedido inválido do usuário**. A rejeição de argumentos pelo Pydantic e a recuperação de uma chamada inválida já foram estudadas anteriormente; aqui não forçamos chamadas artificiais do modelo.

In [ ]:
pmid = "41328123"
doi = "10.7759/cureus.95806"
evaluations = {}

### 01 — Primeiros 5 PMIDs

Esperado: `search_pubmed` com a query atual e `limit=5`. Conferir total encontrado, quantidade retornada e PMIDs usando a observação.

In [ ]:
evaluations["01"] = run_agent(
    "Quantos registros correspondem à query atual e quais são os primeiros 5 PMIDs?"
)

### 02 — Quantidade maior que a amostra inicial

Esperado: `search_pubmed` com `limit=21`. Se houver menos registros disponíveis, informar essa diferença.

In [ ]:
evaluations["02"] = run_agent("Use a query atual e retorne os primeiros 21 PMIDs.")

### 03 — Metadados de um PMID conhecido

Esperado: `get_pubmed_citations` com `[pmid]`. Conferir título, DOI, periódico e ano na observação.

In [ ]:
evaluations["03"] = run_agent(
    f"Consulte no PubMed os metadados do PMID {pmid}. Informe título, DOI, periódico e ano."
)

### 04 — DOI conhecido no Crossref

Esperado: `get_crossref_work` com o DOI informado. Distinguir referências de citações e identificar a fonte.

In [ ]:
evaluations["04"] = run_agent(
    f"Consulte no Crossref o DOI {doi}. Informe as contagens de referências e citações."
)

### 05 — Encadeamento entre fontes

Esperado: busca com `limit=1`, consulta dos metadados do PMID retornado e consulta do DOI retornado no Crossref, quando houver DOI. Os identificadores devem vir das observações anteriores.

In [ ]:
evaluations["05"] = run_agent(
    "Use a query atual para obter somente o primeiro PMID. Consulte seus metadados "
    "no PubMed e, se houver DOI, consulte no Crossref as contagens de referências e citações."
)

### 06 — Busca vazia

Esperado: informar zero resultados e encerrar sem inventar PMIDs.

In [ ]:
def empty_search(query, limit):
    return {
        "status": "success", "source": "pubmed", "query": query,
        "requested_count": limit, "total_results": 0,
        "returned_count": 0, "pmids": [],
    }

with patch.dict(TOOL_REGISTRY["search_pubmed"], {"handler": empty_search}):
    evaluations["06"] = run_agent(
        "Quantos registros correspondem à query atual e quais são os primeiros 5 PMIDs?"
    )

### 07 — Metadados sem DOI

Esperado: informar que o DOI está ausente e encerrar a consulta correspondente ao Crossref.

A cópia abaixo preserva o retorno completo do caso 03 e altera apenas o DOI para simular sua ausência. Execute o caso 03 primeiro.

In [ ]:
metadata = next(entry["result"] for entry in evaluations["03"]["call_log"]
                if entry["tool"] == "get_pubmed_citations")
without_doi = deepcopy(metadata)
for citation in without_doi["citations"]:
    citation["doi"] = None

def metadata_without_doi(pmids):
    return deepcopy(without_doi)

with patch.dict(TOOL_REGISTRY["get_pubmed_citations"], {"handler": metadata_without_doi}):
    evaluations["07"] = run_agent(
        f"Obtenha os metadados do PMID {pmid} no PubMed. Se houver DOI, consulte suas contagens no Crossref."
    )

### 08 — Registro ausente no Crossref

Esperado: explicar que não foi encontrado um registro nessa fonte. Isso não demonstra que a publicação não existe.

In [ ]:
def crossref_not_found(doi):
    return {"status": "not_found", "source": "crossref", "doi": doi}

with patch.dict(TOOL_REGISTRY["get_crossref_work"], {"handler": crossref_not_found}):
    evaluations["08"] = run_agent(f"Consulte no Crossref o DOI {doi} e informe suas contagens.")

### 09 — Quantidade inválida solicitada pelo usuário

Esperado: explicar que a quantidade mínima é 1 e solicitar uma quantidade válida. Conferir que o agente não substitui silenciosamente o pedido por outra quantidade.

Este caso pode ser resolvido sem chamar ferramentas. Ele **não testa a rejeição de uma chamada com argumentos inválidos**; esse teste do executor já foi feito separadamente.

In [ ]:
evaluations["09"] = run_agent("Use a query atual e retorne os primeiros 0 PMIDs.")

### 10 — Timeout externo

Esperado: a observação indica `external_error`; o agente comunica que a consulta falhou e respeita o orçamento. Uma resposta final ausente ou com contagens inventadas deve ser registrada como falha.

In [ ]:
def crossref_timeout(doi):
    raise httpx.ReadTimeout("Timeout simulado para avaliação")

with patch.dict(TOOL_REGISTRY["get_crossref_work"], {"handler": crossref_timeout}):
    evaluations["10"] = run_agent(f"Consulte no Crossref o DOI {doi} e informe suas contagens.")

## Inspeção de um caso

Troque o identificador para revisar cada execução. Os números da resposta devem corresponder à observação daquela execução, pois as APIs reais podem mudar seus dados.

In [ ]:
case_id = "01"
display(evaluations[case_id]["call_log"])
print(evaluations[case_id]["answer"])
print("Parada:", evaluations[case_id]["stop_reason"])

## Registro manual

Preencha a tabela após inspecionar cada caso. Marque **Sim** quando ferramentas/argumentos, uso das observações e encerramento atenderem aos critérios descritos acima. Registre o motivo quando houver falha.

| Caso | Cenário | Passou? | Evidência ou motivo da falha |
|---|---|---|---|
| 01 | Primeiros 5 PMIDs | | |
| 02 | Pedido de 21 PMIDs | | |
| 03 | Metadados por PMID | | |
| 04 | Crossref por DOI | | |
| 05 | Encadeamento entre fontes | | |
| 06 | Busca vazia | | |
| 07 | Publicação sem DOI | | |
| 08 | Registro ausente no Crossref | | |
| 09 | Pedido de quantidade zero | | |
| 10 | Timeout externo | | |

Ao concluir as dez revisões, a taxa de sucesso é **casos aprovados / 10**. Guarde os resultados e os motivos das falhas para comparar a próxima versão do agente. O teste direto de validação Pydantic permanece separado desta avaliação das tarefas do agente.